# Paso 1: Generación de Datos Sintéticos

## ¿Qué son los datos sintéticos?

Los **datos sintéticos** son datos generados artificialmente por ordenador que imitan las características de datos reales. En lugar de utilizar información de pacientes reales (lo cual requeriría permisos y plantearía problemas de privacidad), creamos un conjunto de datos ficticio que sigue los mismos patrones estadísticos que encontraríamos en una clínica real.

## ¿Por qué los usamos aquí?

- **Privacidad**: No necesitamos datos de pacientes reales para desarrollar y probar el sistema.
- **Control**: Podemos definir exactamente cuántos deportistas queremos y qué características tendrán.
- **Reproducibilidad**: Con la misma semilla aleatoria, siempre obtenemos exactamente los mismos datos.
- **Desarrollo**: Nos permiten construir y validar el modelo antes de aplicarlo con datos reales.

Este notebook genera el conjunto de datos base que usaremos en todos los pasos siguientes.

In [1]:
import sys
from pathlib import Path

# Añadir el directorio raíz del proyecto al path
proyecto_raiz = Path("..").resolve()
sys.path.insert(0, str(proyecto_raiz))

from src.generador_datos import generar_dataset
from src.variables import VARIABLES, TOTAL_COLUMNAS
import pandas as pd

## Configuración

Aquí puedes ajustar dos parámetros:

| Parámetro | Descripción | Valor por defecto |
|-----------|-------------|-------------------|
| `N_DEPORTISTAS` | Número de deportistas que tendrá el dataset | 500 |
| `SEMILLA` | Número que controla la aleatoriedad (mismo número = mismos datos siempre) | 42 |

**Nota:** Este notebook genera un dataset de exploración (500 deportistas). El modelo de producción se entrena en `03_entrenamiento_modelo.ipynb` con 5 semillas × 1 000 muestras = 4 826 deportistas.

In [2]:
N_DEPORTISTAS = 500
SEMILLA = 42

df = generar_dataset(n_deportistas=N_DEPORTISTAS, semilla=SEMILLA)
print(f"Dataset generado: {df.shape[0]} deportistas, {df.shape[1]} columnas")

Generando dataset sintético v2.3 con 500 deportistas (semilla=42)...
  [1/5] Bloque contexto...
  [2/5] Bloque fuerza...
  [3/5] Bloque movilidad...
  [4/5] Bloque control...
  [5/5] Inyectando casos frontera...
  Aplicando reglas v2.3 y calculando score de confianza...

Distribución de riesgo:
  bajo            :  26.8 %
  medio           :  24.4 %
  alto            :  44.8 %
  no_concluyente  :   4.0 %

Distribución de confianza:
  alta            :  21.6 %
  media           :  41.2 %
  baja            :  37.2 %

Dataset generado: 500 filas × 36 columnas.

Dataset generado: 500 deportistas, 36 columnas


## Vista previa de los datos

A continuación se muestran los primeros 10 deportistas del dataset. Puedes ver todas las variables que se han generado para cada uno.

In [3]:
df.head(10)

,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,peso_corporal,nivel_actividad,historial_lesional,dolor_percibido_nrs,hooper_index,riesgo_lesion,score_total,confianza_score,confianza_categoria,reglas_activadas
0,230.4,187.7,148.5,193.1,79.9,86.2,101.5,109.2,145.2,137.7,...,61.6,activo,1,0,13,medio,12.0,76.0,media,M7
1,395.3,247.2,136.8,221.7,125.0,136.4,91.6,91.8,186.3,229.3,...,62.7,elite,0,0,17,bajo,17.0,64.0,media,—
2,205.8,247.7,196.6,191.8,93.1,115.1,88.1,81.2,119.2,123.6,...,50.5,activo,2,1,13,medio,11.0,80.0,alta,M7
3,299.5,300.6,140.4,144.0,146.7,147.5,111.0,80.8,210.6,205.8,...,70.7,activo,3,1,21,bajo,6.0,92.0,alta,—
4,376.0,370.5,271.0,294.6,142.8,121.4,100.0,111.7,248.2,222.9,...,80.3,recreacional,0,0,17,medio,11.5,80.0,alta,M7
5,225.9,163.6,126.4,133.3,94.7,114.7,55.2,53.7,116.4,129.0,...,75.1,recreacional,2,1,8,alto,29.5,48.0,baja,"A1_izq,A6_izq,M6,M7"
6,116.5,175.8,92.2,96.0,105.5,113.5,59.3,50.7,114.1,102.0,...,56.6,activo,0,0,15,alto,25.5,56.0,baja,"A1_der,A4"
7,193.9,194.9,133.8,119.0,140.8,92.7,42.7,43.8,129.6,149.5,...,65.4,recreacional,1,0,22,alto,32.5,44.0,baja,—
8,296.8,218.3,278.5,314.0,142.2,173.1,80.5,141.7,262.5,271.0,...,85.4,recreacional,2,0,13,bajo,14.5,72.0,media,—
9,203.9,600.0,137.8,237.1,172.0,199.6,173.6,156.2,305.3,328.7,...,81.7,recreacional,0,8,16,no_concluyente,9.0,88.0,alta,"A1_der,A4"


## Distribución de niveles de riesgo

Una de las columnas más importantes del dataset es el **nivel de riesgo de lesión**. Aquí podemos ver cuántos deportistas caen en cada categoría.

In [4]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Tabla de frecuencias
print("Distribución de niveles de riesgo:")
print("="*40)
distribucion = df["riesgo_lesion"].value_counts().sort_index()
for nivel, cantidad in distribucion.items():
    porcentaje = cantidad / len(df) * 100
    print(f"  {nivel}: {cantidad} deportistas ({porcentaje:.1f}%)")
print("="*40)

# Gráfico de barras
fig, ax = plt.subplots(figsize=(8, 5))

colores = {"Bajo": "#2ecc71", "Medio": "#f39c12", "Alto": "#e74c3c"}
niveles = distribucion.index.tolist()
valores = distribucion.values.tolist()
barras_colores = [colores.get(n, "#3498db") for n in niveles]

barras = ax.bar(niveles, valores, color=barras_colores, edgecolor="white", linewidth=1.5)

# Etiquetas encima de cada barra
for barra, valor in zip(barras, valores):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 5,
        str(valor),
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.set_title("Distribución de niveles de riesgo de lesión", fontsize=14, pad=15)
ax.set_xlabel("Nivel de riesgo", fontsize=12)
ax.set_ylabel("Número de deportistas", fontsize=12)
ax.set_ylim(0, max(valores) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
ruta_fig = proyecto_raiz / "figuras" / "distribucion_riesgo.png"
plt.savefig(ruta_fig, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Figura guardada en: {ruta_fig}")

Distribución de niveles de riesgo:
  alto: 224 deportistas (44.8%)
  bajo: 134 deportistas (26.8%)
  medio: 122 deportistas (24.4%)
  no_concluyente: 20 deportistas (4.0%)
Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/distribucion_riesgo.png


## Estadísticas descriptivas

La tabla siguiente muestra un resumen estadístico de todas las variables numéricas del dataset:

- **count**: número de valores disponibles
- **mean**: media (promedio)
- **std**: desviación estándar (dispersión de los datos)
- **min / max**: valores mínimo y máximo
- **25% / 50% / 75%**: percentiles (el 50% es la mediana)

In [5]:
df.describe().round(2)

,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,single_leg_squat_valgo_izq,single_leg_hop_der,single_leg_hop_izq,edad,peso_corporal,historial_lesional,dolor_percibido_nrs,hooper_index,score_total,confianza_score
count,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,...,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00
mean,289.26,292.39,187.10,188.97,120.90,121.41,86.05,85.32,168.85,168.41,...,1.00,160.96,161.73,29.55,71.18,2.43,1.18,13.93,20.68,63.71
std,131.88,134.14,85.87,89.55,56.18,59.28,45.00,44.28,73.38,72.31,...,0.99,34.44,36.41,8.15,11.77,2.31,1.61,4.90,9.95,16.05
min,80.00,80.00,50.00,50.00,40.00,40.00,25.00,25.00,50.00,50.00,...,0.00,62.50,50.00,18.00,45.00,0.00,0.00,4.00,0.00,24.00
25%,189.12,190.88,121.82,120.50,79.35,74.90,50.95,52.00,112.18,113.92,...,0.00,138.67,136.80,23.00,62.78,1.00,0.00,10.00,12.50,52.00
50%,267.55,259.50,169.00,169.75,110.00,111.30,78.85,75.20,156.15,152.45,...,1.00,162.30,160.80,29.00,70.70,2.00,1.00,14.00,20.50,64.00
75%,370.05,372.17,244.12,242.55,150.22,154.20,110.30,112.22,217.65,219.62,...,2.00,184.10,186.68,35.00,79.12,4.00,1.00,17.00,27.50,76.00
max,600.00,600.00,400.00,400.00,320.00,320.00,230.00,230.00,340.00,340.00,...,3.00,240.00,240.00,58.00,106.10,10.00,9.00,28.00,50.50,100.00


## Guardar datos

Ahora vamos a guardar el dataset generado en un archivo CSV. Este archivo será leído automáticamente por los siguientes notebooks, por lo que es importante ejecutar este paso correctamente.

El archivo se guardará en la carpeta `datos/sinteticos/` dentro del proyecto.

In [6]:
ruta_salida = proyecto_raiz / "datos" / "sinteticos" / "dataset_sintetico.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8")
print(f"Datos guardados en: {ruta_salida}")

Datos guardados en: /Users/__robeerr/Programacion_Local/IntApp v2/datos/sinteticos/dataset_sintetico.csv


## Resumen y siguiente paso

---

**Lo que hemos hecho en este notebook:**

- Generado un dataset sintético con **500 deportistas** y sus variables fisiológicas y de entrenamiento.
- Comprobado que los datos tienen una distribución realista de niveles de riesgo.
- Guardado el dataset en `datos/sinteticos/dataset_sintetico.csv`.

---

**Siguiente paso: ejecuta el notebook `02_exploracion_datos.ipynb`**

En ese notebook exploraremos el dataset en detalle: veremos qué variables están más relacionadas con el riesgo de lesión, detectaremos valores atípicos y prepararemos los datos para el modelo de machine learning.

---

> Si has modificado `N_DEPORTISTAS` o `SEMILLA`, recuerda volver a ejecutar todos los notebooks desde el principio para que los cambios se propaguen correctamente.